![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# Agents and Tools
*Data Science & Machine Learning - Week 8, Day 1*

## How to use this notebook

On Thursday of last week, you gave a model one function, an exchange-rate lookup, and it asked your code to call it. On Friday, you gave it the right passages of a document before it answered. Both times, **you** decided what happened next.

An **agent** decides for itself. It is an LLM in a loop: it reads the question, chooses a tool, reads the result, and chooses again, until it can answer. That small change lets one assistant look up a rate, do the arithmetic, search the company's policies and search the web, in whatever order the question needs.

You will:

1. Write an agent loop by hand, with the `openai` library you already know, so nothing is magic.
2. Build the same agent with **LangChain**, the most widely used framework for this.
3. Give it **web search** with **Tavily**, and the bank's policies through a **retrieval tool** (Friday's RAG, as a tool).
4. Give it **memory** across turns.
5. Use agents responsibly: step limits, cost, permissions and untrusted text.

The **Appendix** has extra material that is not part of the lesson. This afternoon's second notebook, **Multimodal AI**, adds images and speech.

## Table of contents

- [Setup](#Setup)
- [1. From function calling to agents](#1.-From-function-calling-to-agents)
- [2. An agent loop by hand](#2.-An-agent-loop-by-hand)
- [3. The same agent with LangChain](#3.-The-same-agent-with-LangChain)
- [4. Web search with Tavily](#4.-Web-search-with-Tavily)
- [5. Your documents as a tool](#5.-Your-documents-as-a-tool)
- [6. Memory](#6.-Memory)
- [7. Using agents responsibly](#7.-Using-agents-responsibly)
- [Summary](#Summary)
- [Appendix](#Appendix)

## Setup

You need `pip install openai python-dotenv langchain langchain-openai langchain-tavily sentence-transformers chromadb`.

You need two keys in the `.env` file next to this notebook:

```
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...
```

The OpenAI key is the one from last week. For the Tavily key, create a free account at [tavily.com](https://www.tavily.com) (no card needed); the free plan includes 1,000 searches a month, far more than this course needs. As always, `.env` stays out of git.

Running the whole notebook costs about one US cent and uses about 15 Tavily searches. Agents make their own choices, so your runs will not call exactly the same tools as described here.

In [1]:
import os
import json
import time
import logging
import warnings

os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphRecursionError
from sentence_transformers import SentenceTransformer
import chromadb
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
warnings.filterwarnings("ignore", message=".*torch.jit.script.*")
pd.set_option("display.max_colwidth", 120)

load_dotenv()
print("OpenAI key found:", "OPENAI_API_KEY" in os.environ, "| Tavily key found:", "TAVILY_API_KEY" in os.environ)
client = OpenAI()
MODEL = "gpt-5.4-nano"

OpenAI key found: True | Tavily key found: True


## 1. From function calling to agents

Remember how function calling worked on Thursday:

1. You send the question and the description of a tool.
2. The model answers with a **request**: "call `get_exchange_rate` with EUR and USD".
3. Your code runs the function and sends the result back.
4. The model writes the answer.

That was one round. Many questions need several: *How many dollars and pounds are 250 euros?* needs two rates and two multiplications. An **agent** simply repeats steps 2 and 3 until the model stops asking for tools:

[visual: a loop. Question → LLM → "tool call?" — yes → run the tool → add the result to the messages → back to the LLM; no → final answer. A counter on the loop labelled "max steps"]

The LLM is the brain, the tools are its hands, and **your code is the loop that connects them**. Everything an agent framework does is a more convenient version of that loop.

## 2. An agent loop by hand

Two tools for a bank assistant. The exchange rates are fixed so that the results are reproducible; a real tool would call a rates service. The calculator only accepts digits and arithmetic signs, because `eval` on arbitrary text would let the model run any Python code (keep that in mind for section 7).

In [2]:
RATES = {("EUR", "USD"): 1.08, ("EUR", "GBP"): 0.85, ("EUR", "CHF"): 0.94}

def get_exchange_rate(base, target):
    rate = RATES.get((base.upper(), target.upper()))
    return {"rate": rate} if rate else {"error": f"no rate for {base}/{target}"}

def calculate(expression):
    if not set(expression) <= set("0123456789+-*/(). "):
        return {"error": "only numbers and + - * / ( ) are allowed"}
    return {"result": round(eval(expression), 4)}

FUNCTIONS = {"get_exchange_rate": get_exchange_rate, "calculate": calculate}

TOOLS = [
    {"type": "function", "function": {
        "name": "get_exchange_rate",
        "description": "Get today's exchange rate between two currency codes, e.g. EUR and USD.",
        "parameters": {"type": "object",
                       "properties": {"base": {"type": "string"}, "target": {"type": "string"}},
                       "required": ["base", "target"], "additionalProperties": False},
        "strict": True}},
    {"type": "function", "function": {
        "name": "calculate",
        "description": "Evaluate an arithmetic expression such as '250 * 1.08'.",
        "parameters": {"type": "object",
                       "properties": {"expression": {"type": "string"}},
                       "required": ["expression"], "additionalProperties": False},
        "strict": True}},
]

The loop: call the model; if it asks for tools, run each one, append the results with the role `tool`, and call the model again. `max_steps` is the safety net: an agent that never stops asking for tools would otherwise run, and bill, forever.

In [3]:
def run_agent(question, max_steps=5):
    messages = [{"role": "system", "content": "You are a careful banking assistant. Use the tools for rates and arithmetic."},
                {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
        message = response.choices[0].message
        if not message.tool_calls:                       # no more tools: this is the answer
            return message.content
        messages.append(message)
        for call in message.tool_calls:
            arguments = json.loads(call.function.arguments)
            result = FUNCTIONS[call.function.name](**arguments)
            print(f"step {step}: {call.function.name}({arguments}) -> {result}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
    return "Stopped: too many steps."

print(run_agent("I have 250 euros. How many dollars and how many pounds is that?"))

step 1: get_exchange_rate({'base': 'EUR', 'target': 'USD'}) -> {'rate': 1.08}
step 1: get_exchange_rate({'base': 'EUR', 'target': 'GBP'}) -> {'rate': 0.85}


step 2: calculate({'expression': '250 * 1.08'}) -> {'result': 270.0}


step 3: calculate({'expression': '250 * 0.85'}) -> {'result': 212.5}


Using today’s exchange rates:

- **250 EUR → 270.00 USD** (rate: 1 EUR = 1.08 USD)  
- **250 EUR → 212.50 GBP** (rate: 1 EUR = 0.85 GBP)


Read the trace. In the reference run, the model asked for **both rates at once** in step 1 (a model can request several tool calls in one turn), then for **both multiplications** in step 2, and wrote the answer in step 3. Nobody told it that order: it planned it from the tool descriptions. That is the whole idea of an agent, in about fifteen lines of code.

## 3. The same agent with LangChain

Writing the loop yourself is the best way to understand it, but in a real project you also want memory, streaming, limits, many models and ready-made tools. **LangChain** gives you all of that. Its `create_agent` function builds exactly the loop above.

A LangChain tool is a normal Python function with the `@tool` decorator. The function name becomes the tool name, the **docstring becomes the description** the model reads, and the **type hints become the parameters**. So write the docstring for the model: it is how the agent decides when to use the tool.

In [4]:
@tool
def exchange_rate(base: str, target: str) -> dict:
    """Get today's exchange rate between two currency codes, e.g. EUR and USD."""
    return get_exchange_rate(base, target)

@tool
def calculator(expression: str) -> dict:
    """Evaluate an arithmetic expression such as '250 * 1.08'."""
    return calculate(expression)

print(exchange_rate.name, "|", exchange_rate.description)
print(exchange_rate.args)

exchange_rate | Get today's exchange rate between two currency codes, e.g. EUR and USD.
{'base': {'title': 'Base', 'type': 'string'}, 'target': {'title': 'Target', 'type': 'string'}}


In [5]:
agent = create_agent(
    model=f"openai:{MODEL}",
    tools=[exchange_rate, calculator],
    system_prompt="You are a careful banking assistant. Use the tools for rates and arithmetic.",
)

result = agent.invoke({"messages": [{"role": "user", "content": "I have 250 euros. How many Swiss francs is that?"}]})
print(result["messages"][-1].content)

With today’s exchange rate **1 EUR = 0.94 CHF**, **250 EUR ≈ 235.00 CHF**.


`invoke` returns the whole conversation in `result["messages"]`: the question, every tool request, every tool result and the answer. This helper prints it as a trace and adds up the tokens, which we will use for the rest of the notebook:

In [6]:
def show_trace(result):
    tokens_in = tokens_out = 0
    for message in result["messages"]:
        kind = type(message).__name__
        if kind == "AIMessage":
            tokens_in += message.usage_metadata["input_tokens"]
            tokens_out += message.usage_metadata["output_tokens"]
            for call in message.tool_calls:
                print(f"  -> {call['name']}({call['args']})")
        elif kind == "ToolMessage":
            print(f"  <- {str(message.content)[:100]}")
    print(f"  ({tokens_in:,} input and {tokens_out:,} output tokens)")
    print("\nAnswer:", result["messages"][-1].content)

show_trace(result)

  -> exchange_rate({'base': 'EUR', 'target': 'CHF'})
  <- {"rate": 0.94}
  -> calculator({'expression': '250 * 0.94'})
  <- {"result": 235.0}
  (708 input and 73 output tokens)

Answer: With today’s exchange rate **1 EUR = 0.94 CHF**, **250 EUR ≈ 235.00 CHF**.


## 4. Web search with Tavily

The model's knowledge stops at its training date. Ask it something recent:

In [7]:
question = "What is the latest stable version of pandas, and when was it released?"
print(client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": question}]).choices[0].message.content[:400])

As of my latest knowledge (cut off: **2025-08**), the **latest stable pandas release** is **pandas 2.2.3**, released on **2024-09-04**.

If you want, I can also tell you how to verify the current latest stable version directly from PyPI or GitHub.


In the reference run, the model named a version from its training data and pointed out its knowledge cutoff. That is the best case: a model can also state an old version as if it were current. Either way, the answer is out of date. **Tavily** is a search API built for LLMs: it returns the most relevant pages already cleaned into short text, so the agent does not have to read raw HTML. Here is one search on its own:

In [8]:
web_search = TavilySearch(max_results=3)

results = web_search.invoke({"query": "latest pandas release"})
for item in results["results"]:
    print(f"{float(item['score']):.2f}  {item['title'][:70]}\n      {item['url']}")

0.81  Releases · pandas-dev/pandas · GitHub
      https://github.com/pandas-dev/pandas/releases
0.79  pandas - Python Data Analysis Library
      https://pandas.pydata.org
0.78  pandas - Python Data Analysis Library
      https://pandas.pydata.org/community/blog/pandas-3.0.html


Each result has a title, a URL, a relevance score and a snippet of content. Now give the search tool to an agent, with a system prompt that asks for sources:

In [9]:
research_agent = create_agent(
    model=f"openai:{MODEL}",
    tools=[web_search],
    system_prompt="You answer questions with up-to-date information from the web search. Cite the URL you used. Answer in at most 3 sentences.",
)
show_trace(research_agent.invoke({"messages": [{"role": "user", "content": question}]}))

  -> tavily_search({'query': 'latest stable release pandas version released date 2026', 'search_depth': 'advanced', 'topic': 'general'})
  <- {"query": "latest stable release pandas version released date 2026", "follow_up_questions": null, "a
  -> tavily_search({'query': 'pandas 3.0.6 release date Sep 17 2026 latest version', 'search_depth': 'basic', 'topic': 'general'})
  <- {"query": "pandas 3.0.6 release date Sep 17 2026 latest version", "follow_up_questions": null, "answ
  (6,968 input and 139 output tokens)

Answer: The latest stable version of **pandas is 3.0.6**, and it was released on **September 17, 2026**.  
Source: https://pandas.pydata.org (Latest version / Release date shown on the pandas home page).


The agent wrote its own search query, read the results, and in the reference run **searched a second time** to confirm the release date before answering with a source. It also chose options of the search tool itself, such as the search depth and the topic. Notice the token count: search results are long, so a question with web search costs several thousand input tokens, about ten times a plain question (still a fraction of a cent with this model).

## 5. Your documents as a tool

On Friday you built a RAG pipeline: retrieve the passages, then answer. An agent can use retrieval as **one tool among others**, and decide for itself when the question is about the company's documents and when it needs the web.

Here are a few of the bank's policies (invented for this course), indexed in Chroma with the MiniLM model, exactly as on Friday:

In [10]:
POLICIES = [
    ("cards-1", "Lost or stolen cards: a lost or stolen card must be frozen immediately in the app under Card > Freeze. The bank then sends a replacement card for free within 5 working days."),
    ("cards-2", "Card delivery: new cards arrive within 7 working days. If a card has not arrived after 10 working days, the customer can order a free replacement under Card > Replace."),
    ("cards-3", "PIN blocked: after three wrong PIN attempts the card is blocked. The customer can unblock it in the app under Card > PIN > Unblock, or at any of the bank's ATMs."),
    ("transfers-1", "Transfer limits: customers can send up to 10,000 euros per day by SEPA transfer. Instant transfers are limited to 2,000 euros each."),
    ("fx-1", "Foreign currency: card payments in a foreign currency use the Mastercard exchange rate plus a 1.5% fee. The fee is waived for Premium customers."),
    ("ai-1", "AI assistant policy: the support chatbot must start every conversation with 'You are chatting with Nova, the bank's AI assistant.' Staff must never paste customer names, IBANs or card numbers into external AI tools."),
]

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
policies = chromadb.Client().create_collection("bank_policies", configuration={"hnsw": {"space": "cosine"}})
policies.add(ids=[pid for pid, _ in POLICIES], documents=[text for _, text in POLICIES],
             embeddings=embedder.encode([text for _, text in POLICIES], normalize_embeddings=True))

@tool
def search_bank_policies(query: str) -> str:
    """Search the bank's internal policies (cards, transfers, fees, AI use). Use it for any question about how the bank works."""
    result = policies.query(query_embeddings=embedder.encode([query], normalize_embeddings=True), n_results=2)
    return "\n".join(f"[{pid}] {text}" for pid, text in zip(result["ids"][0], result["documents"][0]))

print(search_bank_policies.invoke({"query": "my card never arrived"}))

[cards-2] Card delivery: new cards arrive within 7 working days. If a card has not arrived after 10 working days, the customer can order a free replacement under Card > Replace.
[cards-1] Lost or stolen cards: a lost or stolen card must be frozen immediately in the app under Card > Freeze. The bank then sends a replacement card for free within 5 working days.


The tool returns the two closest policies with their ids, so the agent can cite them. Now an assistant for the bank's staff with **both** tools. The system prompt says when to use which:

In [11]:
STAFF_PROMPT = (
    "You are an assistant for the bank's staff. Use search_bank_policies for questions about the bank, "
    "and the web search for public, recent information. Cite the policy id or the URL you used. Answer in at most 3 sentences."
)
staff_agent = create_agent(model=f"openai:{MODEL}", tools=[search_bank_policies, web_search], system_prompt=STAFF_PROMPT)

for question in ["My card hasn't arrived after 12 working days. What can I do?",
                 "What does the EU AI Act say chatbots must tell users, and does our bank's policy comply?"]:
    print("Question:", question)
    show_trace(staff_agent.invoke({"messages": [{"role": "user", "content": question}]}))
    print("\n" + "-" * 80)

Question: My card hasn't arrived after 12 working days. What can I do?


  -> search_bank_policies({'query': 'replacement card not received after 12 working days what to do'})
  <- [cards-2] Card delivery: new cards arrive within 7 working days. If a card has not arrived after 10 
  (2,882 input and 139 output tokens)

Answer: If your card hasn’t arrived after 12 working days, you can order a **free replacement** in the app/web under **Card > Replace** (cards normally arrive within **7 working days**, and if it hasn’t arrived after **10 working days** you can replace). If the card was **lost or stolen**, **freeze it immediately** in the app under **Card > Freeze** and the replacement is sent for free within **5 working days**. (Policy: **cards-2**, **cards-1**)

--------------------------------------------------------------------------------
Question: What does the EU AI Act say chatbots must tell users, and does our bank's policy comply?


  -> search_bank_policies({'query': 'EU AI Act chatbot must disclose tell users policy'})
  <- [ai-1] AI assistant policy: the support chatbot must start every conversation with 'You are chatting
  -> tavily_search({'query': 'EU AI Act chatbots disclose that users are interacting with an AI system requirements article', 'search_depth': 'advanced', 'topic': 'general', 'time_range': 'year'})
  <- {"query": "EU AI Act chatbots disclose that users are interacting with an AI system requirements art
  -> search_bank_policies({'query': 'Article 50 EU AI Act compliance chatbot disclosure policy'})
  <- [ai-1] AI assistant policy: the support chatbot must start every conversation with 'You are chatting
  (8,645 input and 248 output tokens)

Answer: The EU AI Act (Article 50) requires that chatbots/AI systems intended to interact with people must be designed so users are clearly informed they are interacting with an AI system, unless it’s obvious from the context (source: Baker Botts article on 

In the reference run, the first question used only the policy search and cited `cards-2` (a free replacement after 10 working days). The second used **both** tools: the web for what the AI Act requires, the policies for what the bank does, and then compared them. No code decided that order; the agent read the question and the tool descriptions.

Agents are not always right about their tools. Try a question that needs a live number:

In [12]:
show_trace(staff_agent.invoke({"messages": [{"role": "user", "content":
    "A customer paid 200 dollars abroad with her card. What fee does the bank charge, and what is today's EUR/USD rate?"}]}))

  -> search_bank_policies({'query': 'card transaction abroad fee percentage 200 dollars fee charged'})
  <- [fx-1] Foreign currency: card payments in a foreign currency use the Mastercard exchange rate plus a
  -> tavily_search({'query': 'EUR USD exchange rate today', 'search_depth': 'fast', 'topic': 'finance'})
  <- {'error': ValueError('Error 400: Finance topic is not supported for fast or ultra-fast search_depth.
  -> tavily_search({'query': 'EUR USD exchange rate today', 'search_depth': 'basic', 'topic': 'general'})
  <- {"query": "EUR USD exchange rate today", "follow_up_questions": null, "answer": null, "images": [], 
  (7,775 input and 192 output tokens)

Answer: For an abroad card payment of **$200**, the bank charges **a 1.5% foreign-currency fee** on top of the Mastercard exchange rate (fee waived for **Premium** customers), i.e., **$3.00** on $200. **Today’s EUR/USD rate is about 1.1203** (mid-market figure shown). Sources: **fx-1** and **https://www.investing.com/currencies

The agent finds the fee in `fx-1` every time, but the rate varies from run to run. In one reference run it answered with a link to a quotes page instead of a number, because the search snippets did not contain today's rate. In another it quoted a rate of about 1.12 from a snippet: a number from a web page that you cannot easily check, and that may be minutes or days old. In that run, the first search even failed, because the agent asked Tavily for a combination of options it does not support, and the agent retried with other options. The answer *looks* complete either way. That is why section 7 insists on checking what agents do, and why a real bank would give the agent a proper rates tool instead of a web search.

## 6. Memory

Like the API on Thursday, an agent on its own forgets everything between calls. A **checkpointer** stores the conversation, and a **`thread_id`** says which conversation a call belongs to. `InMemorySaver` keeps the threads in memory; a real application would use a database.

In [13]:
memory_agent = create_agent(model=f"openai:{MODEL}", tools=[exchange_rate, calculator, search_bank_policies],
                            system_prompt="You are a careful banking assistant. Answer in at most 2 sentences.",
                            checkpointer=InMemorySaver())

marta = {"configurable": {"thread_id": "marta"}}
for text in ["Hi, I'm Marta. I want to send 2,500 euros to my landlord today, instantly.",
             "Is that possible? If not, what's the best option?",
             "Thanks. How much was the amount I mentioned, in pounds?"]:
    result = memory_agent.invoke({"messages": [{"role": "user", "content": text}]}, marta)
    print("Marta:", text, "\nAgent:", result["messages"][-1].content, "\n")

other = {"configurable": {"thread_id": "someone-else"}}
print("New thread:", memory_agent.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, other)["messages"][-1].content)

Marta: Hi, I'm Marta. I want to send 2,500 euros to my landlord today, instantly. 
Agent: Hi Marta—can you confirm the landlord’s country and the recipient’s payment details (IBAN/BIC for a bank transfer, or whether you want to use a card/instant transfer), so I can check the available instant option and any fees? Also tell me the recipient’s name and the destination currency (still EUR?), and I’ll guide you on the exact steps. 



Marta: Is that possible? If not, what's the best option? 
Agent: Instant payment is generally possible, but our instant transfer limit is **€2,000 per transaction**, so **€2,500 can’t be sent instantly as one payment**. The best option is usually a **standard SEPA transfer for €2,500** (arrives quickly—timing depends on the cut-off), or you can do **€2,000 instantly + €500 by standard transfer** if you need part of it right away. 



Marta: Thanks. How much was the amount I mentioned, in pounds? 
Agent: €2,500 is about **£2,125** (using an exchange rate of **1 EUR = 0.85 GBP**). 



New thread: I don’t have access to your personal details or account profile, so I can’t see your name. If you share it, I can use it going forward.


Within Marta's thread, the agent remembered the 2,500 euros: in the reference run it explained that instant transfers are limited to 2,000 euros each and suggested splitting the payment, then converted the original amount to pounds in the last turn. In a new thread it knows nothing. Memory is just the stored message list sent again on every call, so long conversations get more expensive with every turn.

Look at the first answer, too. In the reference run, the agent asked Marta for her landlord's IBAN, although the bank's own policy (`ai-1`) says that IBANs must never go into external AI tools. The agent only follows the rules it is given: if a rule matters, it belongs in the system prompt, and it must be tested.

## 7. Using agents responsibly

An agent does more than talk: it **acts**, with your tools and your money. That changes the risks.

### Limit the steps

An agent can loop: search, read, search again, and never finish. LangChain stops it with `recursion_limit` (each model call and each tool call counts as a step). Here is an agent with a limit that is too low for the question:

In [14]:
question = "Look up the card delivery, PIN and transfer policies one by one, then check each against the web."
try:
    result = staff_agent.invoke({"messages": [{"role": "user", "content": question}]}, {"recursion_limit": 3})
    print("Finished within the limit:", result["messages"][-1].content[:200])
except GraphRecursionError as error:
    print("Stopped:", str(error)[:120])

Stopped: Recursion limit of 3 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limi


Set a limit on every agent you deploy, and decide what the user sees when it is reached.

### Count the cost

Every step is a model call, and every tool result is sent back as input tokens. A plain question costs a few hundred tokens; the web-search questions above cost several thousand. Log the tokens of every run, as `show_trace` does, and multiply by your traffic before you launch.

### Give the least power that works

Our tools only **read**: rates, policies, search results. A tool that **writes** (sends an email, makes a transfer, deletes a file) turns a wrong answer into a wrong action. Keep write tools out of agents, or require a person to approve each call (LangChain has a human-in-the-loop middleware for this). And never give an agent a tool like `eval` without restrictions: our calculator rejects everything except digits and arithmetic signs.

### Treat what tools return as untrusted

Web pages and documents can contain instructions aimed at the model, as you saw on Friday. With an agent, a successful injection can trigger **tool calls**, not just a wrong sentence. Prefer trusted sources (Tavily can be limited to chosen domains with `include_domains`), keep a person in the loop for actions, and check the sources the agent cites.

### Check what the agent did, not just what it said

The trace is your audit log. The EUR/USD answer above can look complete whether or not a reliable rate was found; only the trace tells you where each number came from.

## Summary

| Idea | What it means | Code |
|---|---|---|
| Agent | an LLM in a loop that chooses tools until it can answer | the loop in section 2 |
| Tool | a function the model can ask to run; the docstring is its description | `@tool` on a typed function |
| LangChain agent | the loop, ready-made | `create_agent(model=..., tools=[...], system_prompt=...)` |
| Run and trace | the whole conversation, tool calls included | `agent.invoke({"messages": [...]})["messages"]` |
| Web search | recent, sourced information | `TavilySearch(max_results=3)` |
| Retrieval tool | Friday's RAG as one tool among others | a `@tool` that queries Chroma |
| Memory | the conversation stored per thread | `checkpointer=InMemorySaver()`, `{"configurable": {"thread_id": ...}}` |
| Step limit | stops loops | `{"recursion_limit": 10}` |

The rules that matter most:

+ **Write tool descriptions for the model**: they decide when a tool is used.
+ **Always set a step limit**, and log the tokens of every run.
+ **Read-only tools by default**; actions need a person's approval.
+ **Tool results are untrusted text.**
+ **Check the trace**, not only the answer.

In the lab, you build a research assistant for the bank's product team: web search, the bank's documents and a tool of your own, with an evaluation of which tools it chose and what each answer cost.

## Appendix

### A.1 Watching an agent step by step

`stream` returns each step as it happens, which is how chat apps show "Searching the web..." while the agent works:

In [15]:
for update in agent.stream({"messages": [{"role": "user", "content": "How many dollars are 80 euros?"}]}, stream_mode="updates"):
    for node, data in update.items():
        last = data["messages"][-1]
        print(node, "->", getattr(last, "tool_calls", None) or last.content)

model -> [{'name': 'exchange_rate', 'args': {'base': 'EUR', 'target': 'USD'}, 'id': 'call_7c7JVxz6hTBYrbeZYEvi58ob', 'type': 'tool_call'}]
tools -> {"rate": 1.08}


model -> [{'name': 'calculator', 'args': {'expression': '80 * 1.08'}, 'id': 'call_GFfshKICjdTIgM78BAvi83hH', 'type': 'tool_call'}]
tools -> {"result": 86.4}


model -> 80 EUR is **$86.40** (using an exchange rate of **1 EUR = 1.08 USD**).


### A.2 Structured answers from an agent

`response_format` makes the agent return a Pydantic object at the end, as `parse` did on Thursday:

In [16]:
from pydantic import BaseModel

class Conversion(BaseModel):
    amount_eur: float
    currency: str
    rate: float
    converted: float

structured_agent = create_agent(model=f"openai:{MODEL}", tools=[exchange_rate, calculator], response_format=Conversion)
result = structured_agent.invoke({"messages": [{"role": "user", "content": "Convert 120 euros to pounds."}]})
result["structured_response"]

Conversion(amount_eur=120.0, currency='GBP', rate=0.85, converted=102.0)

### A.3 Other agent frameworks

LangChain's agents run on **LangGraph**, a library for building agents as graphs of steps, which you can use directly when you need more control. OpenAI's **Agents SDK**, LlamaIndex and others solve the same problem with different styles. The ideas of this notebook (tools, the loop, memory, limits, traces) are the same in all of them.